# 10 · Converting Data Types & Custom NA Values

Applies the missing-data and type-casting ideas from notebook 09 to a real column
in the survey data (`YearsCode`), which mixes numbers with text answers like
`'Less than 1 year'`.

In [ ]:
import pandas as pd

na_vals = ['NA', 'Missing']
df = pd.read_csv('data/survey_results_public.csv', index_col='Respondent', na_values=na_vals)
schema_df = pd.read_csv('data/survey_results_schema.csv', index_col='Column')
# na_values=na_vals tells pandas "also treat the literal text 'NA' and 'Missing' as NULL/NaN
# while reading the file", instead of leaving them as plain text.
# SQL: LOAD DATA INFILE 'data/survey_results_public.csv' INTO TABLE survey_results_public
#      FIELDS TERMINATED BY ',' ENCLOSED BY '"' LINES TERMINATED BY '\n' IGNORE 1 ROWS;
#      UPDATE survey_results_public SET <col> = NULL WHERE <col> IN ('NA','Missing');   -- per column


In [ ]:
pd.set_option('display.max_columns', 85)   # display settings only, no SQL equivalent
pd.set_option('display.max_rows', 85)


In [ ]:
df.head()
# SQL: SELECT * FROM survey_results_public LIMIT 5;


In [ ]:
df['YearsCode'].head(10)
# SQL: SELECT YearsCode FROM survey_results_public LIMIT 10;


In [ ]:
df['YearsCode'].unique()
# Shows every distinct value pandas found in this column — useful for spotting
# non-numeric entries (like 'Less than 1 year') before trying to do math on it.
# SQL: SELECT DISTINCT YearsCode FROM survey_results_public;


In [ ]:
df['YearsCode'].replace('Less than 1 year', 0, inplace=True)
# SQL: UPDATE survey_results_public SET YearsCode = 0 WHERE YearsCode = 'Less than 1 year';


In [ ]:
df['YearsCode'].replace('More than 50 years', 51, inplace=True)
# SQL: UPDATE survey_results_public SET YearsCode = 51 WHERE YearsCode = 'More than 50 years';


In [ ]:
df['YearsCode'] = df['YearsCode'].astype(float)
# Now that every value is a plain number (as text), convert the whole column to a numeric type.
# SQL: ALTER TABLE survey_results_public MODIFY COLUMN YearsCode FLOAT;


In [ ]:
df['YearsCode'].mean()
# SQL: SELECT AVG(YearsCode) FROM survey_results_public;


In [ ]:
df['YearsCode'].median()
# SQL: MySQL has no built-in MEDIAN(), so it needs a window-function subquery:
# SELECT AVG(YearsCode) FROM (
#     SELECT YearsCode,
#            ROW_NUMBER() OVER (ORDER BY YearsCode) AS rn,
#            COUNT(*)     OVER ()                   AS cnt
#     FROM survey_results_public
#     WHERE YearsCode IS NOT NULL
# ) ranked
# WHERE rn IN (FLOOR((cnt+1)/2), FLOOR((cnt+2)/2));
